# 03 · Paired YOLO26n partial fine-tuning

**Inputs:** frozen protocol/subsets/policy/decision, matched training exports,
Raw development export and the pinned local pretrained checkpoint.
**Outputs:** named run directories, best/last native checkpoints, histories,
freeze/gradient audits, epoch costs and completion/resume receipts.

Set paths and resume choices below, then **Restart & Run All**. Only training runs;
this notebook never prepares missing data or launches preprocessing/baseline inference.
Both branches use the same initialization, IDs/labels, budget and Raw-development
selection. The architecture supplies the backbone boundary. Training itself includes
native per-epoch development validation. Final-test pixels remain locked.

The refactored implementation gets a new identity/root. Existing S10 evidence is
inspected by notebook 04 using its original receipts, never relabeled or resumed under
changed code. Interrupted runs resume only under identical code/data/settings.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import read_json, require_paths
from vehicle_detection.finetuning import TRAIN_SETTINGS, prepare_pair, train_branch
from vehicle_detection.protocol import validate_protocol

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
PROTOCOL_PATH = ROOT / "runs/data-v1/protocol.json"
PREPARED = ROOT / "runs/preprocessing-v1"
RAW_DEVELOPMENT = ROOT / "runs/data-v1/datasets/raw"
CHECKPOINT = ROOT / "weights/yolo26n.pt"
TRAIN_ROOT = ROOT / "runs/training-s10-v1"
DECISION_PATH = ROOT / "docs/issue11-v2/v3_decision.json"
RESUME = {"raw": False, "adaptive": False}
require_paths(
    protocol=PROTOCOL_PATH,
    subsets=PREPARED / "subsets.json",
    policy=PREPARED / "policy.json",
    decision=DECISION_PATH,
    training_pair=PREPARED / "datasets/S10/pair.json",
    development=RAW_DEVELOPMENT / "verified.json",
    checkpoint=CHECKPOINT,
)
protocol = validate_protocol(read_json(PROTOCOL_PATH))

In [ ]:
experiment = prepare_pair(
    protocol,
    read_json(PREPARED / "subsets.json"),
    read_json(PREPARED / "policy.json"),
    read_json(DECISION_PATH),
    PREPARED / "datasets/S10",
    RAW_DEVELOPMENT,
    CHECKPOINT,
    TRAIN_ROOT,
)
print("Experiment identity:", experiment["experiment_id"])
print("Backbone modules frozen:", experiment["freeze"])
display(pd.Series(TRAIN_SETTINGS, name="identical branch settings"))

In [ ]:
completed = {
    branch: train_branch(experiment, branch, TRAIN_ROOT, resume=RESUME[branch])
    for branch in ("raw", "adaptive")
}
print("Completed paired budgets:", {b: r["budget"] for b, r in completed.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for branch in ("raw", "adaptive"):
    history = pd.read_csv(TRAIN_ROOT / branch / "results.csv")
    history.plot(x="epoch", y="metrics/mAP50-95(B)", ax=axes[0], label=branch)
    history.plot(x="epoch", y="train/box_loss", ax=axes[1], label=branch)
    audit = read_json(TRAIN_ROOT / branch / "freeze_audit.json")
    print(branch, "frozen/trainable:", audit["frozen_count"], audit["trainable_count"])
axes[0].set_title("Shared Raw development validation")
axes[1].set_title("Native training box loss")
plt.tight_layout()
plt.show()